# NBA Oracle Pipeline — Wire Trained Model into TF (Colab T4)

**One-shot pipeline**: pulls Kaggle Oracle pkl + feature cache, computes 2025-26 game predictions, pushes JSON to NBA TF Space, triggers Day-0 reset.

**Inputs (HF datasets):**
- `LBJLincoln26/nba-oracle-model` — trained RF + isotonic calibrator (CV Brier 0.22087, best fold 0.21383)
- `LBJLincoln26/nba-feature-cache` — pre-computed 6509x6452 feature matrix

**Output:** `data/oracle-predictions-latest.json` on `LBJLincoln26/nba-llm-trading-floor` (NEW path, no LFS, ~2MB).

**Setup before running:** Runtime → CPU is fine (no training, just inference). Add HF_TOKEN to Colab Secrets (must have write access to `LBJLincoln26/*`).

**Time:** ~3 min total, no GPU needed.

## 1. Install + secrets

In [ ]:
!pip install --quiet huggingface_hub scikit-learn==1.6.1
import os
from google.colab import userdata
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = input('Paste HF_TOKEN with LBJLincoln26 write access: ').strip()
os.environ['HF_TOKEN'] = HF_TOKEN
print('token set, len=', len(HF_TOKEN))

## 2. Pull Oracle pkl + feature cache from HF

In [ ]:
from huggingface_hub import hf_hub_download

PKL_PATH = hf_hub_download('LBJLincoln26/nba-oracle-model', 'nba-oracle.pkl', repo_type='dataset', token=HF_TOKEN)
NPZ_PATH = hf_hub_download('LBJLincoln26/nba-feature-cache', 'nba_cached_data.npz', repo_type='dataset', token=HF_TOKEN)
GAMES_PATH = hf_hub_download('LBJLincoln26/nba-llm-trading-floor', 'data/games-2025-26.json', repo_type='space', token=HF_TOKEN)
FULLODDS_PATH = hf_hub_download('LBJLincoln26/nba-llm-trading-floor', 'data/full-odds-2025-26.json', repo_type='space', token=HF_TOKEN)
print('pkl:', PKL_PATH)
print('npz:', NPZ_PATH)
print('games:', GAMES_PATH)
print('full-odds:', FULLODDS_PATH)

## 3. Load + slice + predict

In [ ]:
import pickle, json
import numpy as np

with open(PKL_PATH, 'rb') as f:
    bundle = pickle.load(f)
model = bundle['model']
calibrator = bundle.get('calibrator')
feature_indices = bundle['feature_indices']
print(f'oracle: {type(model).__name__} CV Brier={bundle["cv_brier_mean"]:.5f} n_features={len(feature_indices)}')
print(f'fold-by-fold Brier: {bundle["cv_brier_per_fold"]}')

data = np.load(NPZ_PATH, allow_pickle=True)
X = data['X']; y = data['y']
print(f'cache: X.shape={X.shape} y.shape={y.shape} y.mean={y.mean():.3f}')

X_oracle = X[:, feature_indices]
p_raw = model.predict_proba(X_oracle)[:, 1]
if calibrator is not None:
    p_home_all = calibrator.predict(p_raw)
    print(f'calibrated: raw mean={p_raw.mean():.3f} → cal mean={p_home_all.mean():.3f}')
else:
    p_home_all = p_raw

from sklearn.metrics import brier_score_loss
print(f'in-sample Brier: {brier_score_loss(y, p_home_all):.5f}')
print(f'p_home distribution: min={p_home_all.min():.3f} median={np.median(p_home_all):.3f} max={p_home_all.max():.3f}')

## 4. Build chronological game-key list (matches cache row order)

The cache was built by `scripts/karpathy/build_cache.py` which runs `engine.build()` on chronologically-loaded historical games. We replicate the same load order here to map cache row → game_key.

In [ ]:
import urllib.request

GH_REPO = 'https://raw.githubusercontent.com/LBJLincoln/mon-ipad/main'
SEASONS = ['2017-18','2018-19','2019-20','2020-21','2021-22','2022-23','2023-24','2024-25','2025-26']
all_games = []
for s in SEASONS:
    try:
        u = f'{GH_REPO}/data/historical/games-{s}.json'
        raw = json.loads(urllib.request.urlopen(u, timeout=30).read())
        gs = raw if isinstance(raw, list) else raw.get('games', [])
        all_games.extend(gs)
        print(f'  {s}: {len(gs)} games')
    except Exception as e:
        print(f'  {s} FAIL: {e}')
print(f'total: {len(all_games)} games loaded')

def _abbr(v):
    if isinstance(v, dict): return v.get('team_abbr') or ''
    if isinstance(v, str): return v
    return ''

game_keys = []
for g in all_games:
    date = str(g.get('game_date',''))[:10]
    home = _abbr(g.get('home_team') or g.get('home'))
    away = _abbr(g.get('away_team') or g.get('away'))
    if not (date and home and away):
        m = (g.get('matchup') or '').replace(' ','')
        if '@' in m:
            parts = m.split('@')
            if len(parts)==2: away,home = parts
    game_keys.append({'date': date, 'home': home, 'away': away, 'game_id': g.get('game_id','')})
print(f'game_keys built: {len(game_keys)}')

## 5. Map cache rows → 2025-26 game keys

Cache has 6509 rows; we have 11532 games. `engine.build()` processes them in order and may skip some. Strategy: take the LAST `len(X)` game_keys (cache always grows by appending recent games).

In [ ]:
if len(game_keys) < len(X):
    raise RuntimeError(f'Have {len(game_keys)} keys but cache has {len(X)} rows')

skip = len(game_keys) - len(X)
print(f'skipping first {skip} games (no rolling-window context), mapping last {len(X)} 1:1')
matched_keys = game_keys[skip:]

rows_2526 = [(i, k) for i, k in enumerate(matched_keys) if str(k['date']).startswith(('2025','2026'))]
print(f'2025-26 rows in cache: {len(rows_2526)}')
if rows_2526:
    print(f'first: row {rows_2526[0][0]} = {rows_2526[0][1]}')
    print(f'last:  row {rows_2526[-1][0]} = {rows_2526[-1][1]}')

## 6. Build TF predictions JSON (per-game + per-category from full-odds)

In [ ]:
import math
SIGMA_MARGIN = 12.0
SIGMA_TOTAL = 18.0
def _ncdf(x): return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

full_odds = json.load(open(FULLODDS_PATH))
print(f'full-odds covers {len(full_odds)} games (real-odds source)')

out = {}
for row_i, k in rows_2526:
    p_home = float(p_home_all[row_i])
    gk = f"{k['date']}_{k['away']}@{k['home']}"
    margin = (p_home - 0.5) * 8.0
    total = 224.0
    per_cat = {}
    fo = full_odds.get(gk) or {}
    cats = (fo.get('categories') or fo) if isinstance(fo, dict) else {}
    if isinstance(cats, dict):
        for cat_name, ci in cats.items():
            try:
                odds = float((ci or {}).get('odds') or 0)
            except: continue
            if odds <= 1.001: continue
            market_p = 1.0 / odds
            line = (ci or {}).get('line', 0) or 0
            mp = None
            if cat_name == 'ml_home': mp = p_home
            elif cat_name == 'ml_away': mp = 1 - p_home
            elif cat_name == 'spread_home' or cat_name.startswith('alt_spread_home'):
                mp = _ncdf((margin + line) / SIGMA_MARGIN)
            elif cat_name == 'spread_away' or cat_name.startswith('alt_spread_away'):
                mp = _ncdf((-margin - line) / SIGMA_MARGIN)
            elif cat_name == 'total_over' or cat_name.startswith('alt_total_over'):
                mp = 1 - _ncdf((line - total) / SIGMA_TOTAL) if line else None
            elif cat_name == 'total_under' or cat_name.startswith('alt_total_under'):
                mp = _ncdf((line - total) / SIGMA_TOTAL) if line else None
            if mp is None or not (0 < mp < 1): continue
            per_cat[cat_name] = {'prob': round(mp,4), 'edge': round(mp-market_p,4)}
            if line: per_cat[cat_name]['line'] = line
    out[gk] = {
        'game_key': gk,
        'derived_core': {
            'predicted_p_home': round(p_home, 4),
            'predicted_margin': round(margin, 2),
            'predicted_total': total,
        },
        'per_category': per_cat,
        'category_count': len(per_cat),
        'oracle_brier_cv': bundle['cv_brier_mean'],
        'oracle_n_features': len(feature_indices),
        'consensus_ml_direction': 'home' if p_home > 0.5 else 'away',
        'consensus_ml_confidence': round(abs(p_home - 0.5), 4),
    }
print(f'built predictions for {len(out)} games')
json.dump(out, open('/tmp/oracle_predictions.json','w'))
import os
print(f'size: {os.path.getsize("/tmp/oracle_predictions.json")/1024/1024:.2f} MB')

## 7. Push predictions to NBA TF Space (NEW path, no LFS)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
TF = 'LBJLincoln26/nba-llm-trading-floor'
ts = __import__('datetime').datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')

r = api.upload_file(
    path_or_fileobj='/tmp/oracle_predictions.json',
    path_in_repo='data/oracle-predictions-latest.json',
    repo_id=TF, repo_type='space',
    commit_message=f'[ORACLE] Wired Kaggle Oracle predictions ({ts}) — CV Brier {bundle["cv_brier_mean"]:.5f} best fold {min(bundle["cv_brier_per_fold"]):.5f} 200 features'
)
print('uploaded:', r)

## 8. Day-0 reset + restart NBA TF (under Oracle predictions)

In [ ]:
from huggingface_hub import CommitOperationDelete
import time

# Pause
try:
    api.pause_space(TF)
    print('paused')
except Exception as e: print('pause:', e)
time.sleep(8)

# Wipe state
files = api.list_repo_files(TF, repo_type='space')
wipe = [f for f in files if (f.startswith('data/decisions/day-')
        or f == 'data/runtime/state.json'
        or f == 'data/runtime/agent_logs.json'
        or f == 'data/runtime/council_plans.json')]
if wipe:
    ops = [CommitOperationDelete(path_in_repo=f) for f in wipe]
    r = api.create_commit(repo_id=TF, repo_type='space', operations=ops,
                          commit_message=f'[ORACLE] Day-0 wipe ({len(wipe)} files) before Oracle launch')
    print(f'wiped {len(wipe)} files')

# Restart (POST /api/restart unpauses + factory_reboot)
import urllib.request
req = urllib.request.Request(
    f'https://huggingface.co/api/spaces/{TF}/restart',
    method='POST',
    headers={'Authorization': f'Bearer {HF_TOKEN}', 'Content-Type': 'application/json'},
    data=b'{}'
)
with urllib.request.urlopen(req, timeout=20) as resp:
    print('restart:', resp.read().decode()[:200])

## Done.

TF will rebuild + restart. Within ~3 min the new Oracle predictions are live and the agents start Day-0 under the real model.

Verify: https://lbjlincoln26-nba-llm-trading-floor.hf.space/api/leaderboard